Em ơi, tổng hợp nhanh giúp anh: năm 2021 mình bán được bao nhiêu đơn hàng trên toàn chain? Anh cần con số này cho slide đầu tiên của bài thuyết trình sáng mai. 1 số thôi, không cần chi tiết.

In [1]:
SELECT 
    COUNT( DISTINCT [order_number]) AS sum_order
FROM [retails].[sales]
WHERE YEAR([order_date]) = '2021'



(1 row affected)

Total execution time: 00:00:00.071

sum_order
498


<span style="font-size: 14px;">Em liệt kê giúp chị toàn bộ category sản phẩm mình đang bán, kèm số SKU trong mỗi category. Chị cần để làm product mix overview cho slide marketing. Sắp theo alphabet.</span>

In [2]:
SELECT 
    [category]
    , COUNT([product_key]) AS sum_category_number
FROM [retails].[products]
GROUP BY [category]
ORDER BY [category]

(8 rows affected)

Total execution time: 00:00:00.024

category,sum_category_number
Audio,115
Cameras and camcorders,372
Cell phones,285
Computers,606
Games and Toys,166
Home Appliances,661
"Music, Movies and Audio Books",90
TV and Video,222


<span style="font-size: 14px;">Anh cần top 10 thành phố có nhiều khách hàng nhất để team CRM chọn địa điểm tổ chức event tri ân. Gửi anh: tên city, state, country, và số lượng customer</span>

In [3]:

SELECT 
    TOP 10
    cs.[city]
    , cs.[state]
    , cs.[country]
    , COUNT( DISTINCT CS.[customer_key]) AS total_customer
FROM [retails].[customers] cs
GROUP BY 
    cs.[city]
    , cs.[state]
    , cs.[country]
ORDER BY total_customer DESC

(10 rows affected)

Total execution time: 00:00:00.083

city,state,country,total_customer
Toronto,Ontario,Canada,203
New York,New York,United States,130
Los Angeles,California,United States,119
Montreal,Quebec,Canada,97
Chicago,Illinois,United States,90
Houston,Texas,United States,84
Calgary,Alberta,Canada,70
Dallas,Texas,United States,67
Vancouver,British Columbia,Canada,59
Philadelphia,Pennsylvania,United States,53


Em ơi, chị đang làm P&L tháng 12/2020 cho board meeting. Cần tổng doanh thu tháng 12/2020 <span style="color: var(--vscode-foreground);">trên toàn bộ stores — là số tiền khách&nbsp;</span> <span style="color: var(--vscode-foreground);">đã trả (giá bán × số lượng), không phải lợi nhuận. Gửi chị 1 con số USD trước 5pm hôm nay.</span>

In [4]:
-- Đi tính tổng doanh thu trên từng đơn hàng 
WITH data_raw AS ( -- Tính số tiền chi tiết cho từng phân loại trên mỗi đơn hàng
    SELECT 
        s.order_number
        , sum(p.unit_price_usd * s.quantity) AS total_invoice_amount
    FROM retails.sales s
    INNER JOIN retails.products p
        ON s.product_key = p.product_key
    WHERE 1 =1 
        AND s.order_date BETWEEN '2020-12-01' AND '2020-12-31'
    GROUP BY
        s.order_number
)
SELECT 
    sum(total_invoice_amount) AS total_revenue_month12
FROM data_raw


(1 row affected)

Total execution time: 00:00:00.041

total_revenue_month12
651526.44


<span style="font-size: 14px;">Mình đang có bao nhiêu store ở từng quốc gia vậy em? Cho anh 1 bảng đơn giản: country + số store. Sắp theo số store giảm dần để thấy nước nào đang dominate.</span>

In [5]:
SELECT 
    s.country
    , COUNT(*) AS total_store    
FROM retails.stores s
WHERE s.state != 'Online'
GROUP BY s.country
ORDER BY total_store DESC



(8 rows affected)

Total execution time: 00:00:00.019

country,total_store
United States,24
Germany,9
United Kingdom,7
France,7
Australia,6
Canada,5
Netherlands,5
Italy,3


<span style="font-size: 14px;">Em ơi, team merchandising đang lập kế hoạch restock Q1. Anh cần biết top 5 sản phẩm bán chạy nhất trong mỗi category để quyết định nhập hàng. 'Bán chạy' = tổng số lượng bán ra từ trước đến nay. Output 1 bảng: mỗi dòng 1 sản phẩm, có category, tên sản phẩm, tổng lượng bán, và thứ hạng (1-5) trong category đó.</span>

In [6]:
WITH data_raw AS (
    SELECT
        s.product_key
        , SUM(s.quantity) AS sum_quantity_product
    FROM retails.sales s
    GROUP BY s.product_key
),
ranked_products AS (
    SELECT
        p.category
        , p.product_name
        , d.sum_quantity_product
        , DENSE_RANK() OVER (PARTITION BY p.category ORDER BY d.sum_quantity_product DESC) AS rank_quantity
    FROM retails.products p
    INNER JOIN data_raw d
        ON d.product_key = p.product_key
)
SELECT
    category
    , product_name
    , sum_quantity_product
    , rank_quantity
FROM ranked_products
WHERE rank_quantity <= 5
ORDER BY category, rank_quantity

(46 rows affected)

Total execution time: 00:00:00.066

category,product_name,sum_quantity_product,rank_quantity
Audio,WWI 1GB Digital Voice Recorder Pen E100 Black,431,1
Audio,WWI 1GB Digital Voice Recorder Pen E100 Red,388,2
Audio,WWI 4GB Video Recording Pen X200 Yellow,370,3
Audio,WWI 4GB Video Recording Pen X200 Black,366,4
Audio,WWI Wireless Bluetooth Stereo Headphones M270 Black,359,5
Cameras and camcorders,Contoso Cyber Shot Digital Cameras Adapter E306 Black,101,1
Cameras and camcorders,Contoso Digital Camera/Camcorder USB Cable E324 Purple,97,2
Cameras and camcorders,Contoso Macro Zoom Lens X300 Black,95,3
Cameras and camcorders,Contoso Carrying Case E312 Pink,95,3
Cameras and camcorders,Contoso General Soft Carrying Case E318 Black,93,4


Chị cần biết margin gross trung bình của từng subcategory để đánh giá pricing strategy. Margin % = (giá bán − giá vốn) / giá bán. Chỉ xét subcategory có ít nhất 10 sản phẩm để số có ý nghĩa. Sắp theo margin giảm dần.

In [7]:
SELECT 
    p.subcategory
    , ROUND( AVG((p.unit_price_usd - p.unit_cost_usd) * 100/p.unit_price_usd), 2) AS margin_product
FROM retails.products p
GROUP BY p.subcategory
HAVING COUNT(p.product_key) >= 10
ORDER BY margin_product DESC

(32 rows affected)

Total execution time: 00:00:00.027

subcategory,margin_product
Digital SLR Cameras,60.4400000000000
Digital Cameras,57.3900000000000
Projectors & Screens,57.3300000000000
Movie DVD,57.2400000000000
Monitors,56.7100000000000
"Printers, Scanners & Fax",56.3800000000000
Camcorders,56.2600000000000
Smart phones & PDAs,55.9300000000000
Bluetooth Headphones,55.6900000000000
Touch Screen Phones,55.2600000000000


Em giúp anh check SLA giao hàng nhé. Lưu ý: chỉ những đơn có phát sinh giao hàng tới tay khách mới có ngày giao — đơn mua trực tiếp tại cửa hàng thì không. Với nhóm đơn có giao đó, trung bình từ lúc khách đặt đến lúc nhận hàng là bao nhiêu ngày, tính theo từng quốc gia của khách? Gửi anh: tên nước, số đơn có giao, số ngày trung bình — sắp theo số ngày trung bình giảm dần để anh thấy nước nào đang giao chậm nhất.

In [8]:

SELECT 
    c.country
    , COUNT(DISTINCT s.order_number) AS count_order
    , AVG(DATEDIFF( DAY, s.order_date, s.delivery_date)) AS avg_delivery_date
FROM retails.sales s
INNER JOIN retails.customers c
    ON s.customer_key = c.customer_key
WHERE delivery_date IS NOT NULL
GROUP BY c.country
ORDER BY avg_delivery_date DESC

(8 rows affected)

Total execution time: 00:00:00.096

country,count_order,avg_delivery_date
Australia,281,4
Canada,512,4
France,133,4
Germany,538,4
Italy,211,4
Netherlands,200,4
United Kingdom,637,4
United States,3068,4


Chị cần chuẩn bị danh sách khách VIP theo từng quốc gia để team CRM làm campaign tri ân cuối năm. Ở mỗi nước, cho chị 1 khách hàng chi tiêu nhiều nhất trong năm 2020. Output: country, tên khách, tổng chi tiêu 2020.

In [9]:
WITH data_raw_total_spending AS( -- tính tổng doanh thu theo từng khách hàng
    SELECT 
        s.customer_key
        , SUM(s.quantity * p.unit_price_usd) AS total_spending
    FROM retails.sales s
    INNER JOIN retails.products p
        ON p.product_key = s.product_key
    WHERE YEAR(s.order_date) = 2020
    GROUP BY s.customer_key
), 
data_raw_rank_country AS ( 
    SELECT
        c.country
        , c.name
        , d.total_spending
        , DENSE_RANK() OVER ( PARTITION BY c.country ORDER BY d.total_spending DESC) AS rank_spending_country 
    FROM data_raw_total_spending d
    INNER JOIN retails.customers c
        ON c.customer_key = d.customer_key
)
SELECT
    country
    , name
    , total_spending
FROM data_raw_rank_country
WHERE rank_spending_country = 1

(8 rows affected)

Total execution time: 00:00:00.095

country,name,total_spending
Australia,Makayla Hassall,16778.46
Canada,Zane Belgrave,20644.89
France,Alice Lafond,18659.88
Germany,Daniel Kaestner,17931.00
Italy,Gaspare Trevisan,13831.89
Netherlands,Polat Klarenbeek,13369.52
United Kingdom,Dominic Banks,19371.06
United States,Mie Huus,33275.47


Merchandising nghi có sản phẩm nằm trong catalog nhưng chưa từng bán được cái nào — zombie inventory. Em list giúp anh toàn bộ SKU đó để team clearance xử lý. Gửi anh: mã sản phẩm, tên sản phẩm, brand, category.

In [10]:
SELECT 
    p.product_key
    , p.product_name
    , p.brand
    , p.category
FROM retails.products p
LEFT JOIN retails.sales s 
    ON s.product_key = p.product_key
WHERE s.product_key IS NULL

(25 rows affected)

Total execution time: 00:00:00.104

product_key,product_name,brand,category
274,Contoso Home Theater System 5.1 Channel M1530 White,Contoso,TV and Video
1885,Contoso Washer & Dryer 21in E210 Green,Contoso,Home Appliances
2190,Adventure Works Floor Lamp X1150 Black,Adventure Works,Home Appliances
2191,Adventure Works Floor Lamp M2150 Black,Adventure Works,Home Appliances
2193,Adventure Works Chandelier M6150 Black,Adventure Works,Home Appliances
2211,Adventure Works Wall Lamp E3150 Silver,Adventure Works,Home Appliances
2212,Adventure Works Desk Lamp E1300 Silver,Adventure Works,Home Appliances
2214,Adventure Works Floor Lamp X1150 Grey,Adventure Works,Home Appliances
2219,Adventure Works Wall Lamp E3150 Grey,Adventure Works,Home Appliances
2220,Adventure Works Desk Lamp E1300 Grey,Adventure Works,Home Appliances


Chị cần slide cho CEO presentation show doanh thu từng tháng + doanh thu tích luỹ từ đầu period 24 tháng gần nhất. Tháng-doanh thu để nhìn trend, tích luỹ để nhìn scale. Xuất 1 bảng: year\_month, doanh thu tháng đó, doanh thu cộng dồn.

In [11]:
WITH data_raw_1 AS ( 
    SELECT
        FORMAT( s.order_date, 'yyyy - MM') AS YearMonth 
        , DATEFROMPARTS( YEAR(s.order_date), MONTH(s.order_date), 1) AS MonthStart
        , SUM( s.quantity * p.unit_price_usd) AS total_revenue_bill
    FROM retails.sales s
    INNER JOIN retails.products p
        ON p.product_key = s.product_key
    WHERE s.order_date >= DATEADD( MONTH, -24, (SELECT MAX(order_date) FROM retails.sales))
    GROUP BY FORMAT( s.order_date, 'yyyy - MM'), DATEFROMPARTS( YEAR(s.order_date), MONTH(s.order_date), 1) 
)
SELECT 
    YearMonth
    , total_revenue_bill
    , SUM(total_revenue_bill) OVER( ORDER BY MonthStart) AS cumulative_revenue
FROM data_raw_1 d1
ORDER BY MonthStart

(25 rows affected)

Total execution time: 00:00:02.801

YearMonth,total_revenue_bill,cumulative_revenue
2019 - 02,829119.85,829119.85
2019 - 03,845925.09,1675044.94
2019 - 04,149892.71,1824937.65
2019 - 05,1594446.47,3419384.12
2019 - 06,1404861.40,4824245.52
2019 - 07,1408714.62,6232960.14
2019 - 08,1500784.77,7733744.91
2019 - 09,1547870.73,9281615.64
2019 - 10,1575168.82,10856784.46
2019 - 11,1715659.88,12572444.34


Anh đang nghi có vấn đề về retention nhưng chưa có proof. Em giúp anh: khách lần đầu mua năm X có bao nhiêu % quay lại mua ở năm X+1, X+2, X+3? Gom khách theo năm họ mua đầu tiên. Xuất matrix: năm mua đầu × số năm sau (0, 1, 2, 3) × % retention. Anh cần báo cáo cho board meeting quarter sau.

In [12]:
WITH _year_purchase AS ( 
    SELECT 
        DISTINCT s.customer_key
        , YEAR(MIN(s.order_date)) AS year_start_buy
    FROM retails.sales s
    GROUP BY s.customer_key
), other_year AS ( 
    SELECT
        DISTINCT yp.customer_key
        , yp.year_start_buy 
        , YEAR(s.order_date) AS other_year
        , YEAR(s.order_date) - yp.year_start_buy AS number_client_year
    FROM _year_purchase yp
    INNER JOIN retails.sales s
        ON s.customer_key = yp.customer_key
), cohort_analysis AS ( 
    SELECT 
        ot.year_start_buy
        , ot.number_client_year
        , COUNT( DISTINCT ot.customer_key) AS count_customer
    FROM other_year ot
    GROUP BY ot.year_start_buy, ot.number_client_year
), cohort_size AS ( 
    SELECT 
        ca.year_start_buy
        , ca.count_customer AS total_customer
    FROM cohort_analysis ca
    WHERE ca.number_client_year = 0
)

SELECT 
    cs.year_start_buy
    , ROUND( 100.0 * MAX( CASE WHEN ca.number_client_year = 1 THEN ca.count_customer END) / cs.total_customer, 2) AS  retention_1yr
    , ROUND( 100.0 * MAX( CASE WHEN ca.number_client_year = 2 THEN ca.count_customer END) / cs.total_customer, 2) AS  retention_2yr
    , ROUND( 100.0 * MAX( CASE WHEN ca.number_client_year = 3 THEN ca.count_customer END) / cs.total_customer, 2) AS  retention_3yr
FROM cohort_size cs
INNER JOIN cohort_analysis ca
    ON cs.year_start_buy = ca.year_start_buy
GROUP BY cs.year_start_buy, cs.total_customer
ORDER BY cs.year_start_buy

Warning: Null value is eliminated by an aggregate or other SET operation.

(6 rows affected)

Total execution time: 00:00:00.397

year_start_buy,retention_1yr,retention_2yr,retention_3yr
2016,20.730000000000,32.760000000000,45.650000000000
2017,34.600000000000,46.510000000000,27.060000000000
2018,44.810000000000,27.290000000000,3.640000000000
2019,25.420000000000,2.860000000000,NULL
2020,3.590000000000,NULL,NULL
2021,NULL,NULL,NULL


Em tính giúp chị doanh thu trên mỗi mét vuông của từng store (YTD 2023) để so sánh hiệu quả sử dụng diện tích. Sau đó xếp hạng 4 nhóm (quartile) trong cùng quốc gia — store nào top 25%, store nào bottom 25%. Xuất: store\_key, country, doanh thu/m², quartile trong nước.

In [ ]:
WITH total_revenue AS ( 
    SELECT 
        s.store_key
        , sum(s.quantity * p.unit_price_usd) AS order_value
    FROM retails.sales s
    INNER JOIN retails.products p
        ON s.product_key = p.product_key
    WHERE YEAR(s.order_date) = 2023 -- YTD 2023
    GROUP BY s.store_key
), revenue_acreage AS ( 
    SELECT 
        st.store_key
        , st.country
        , tr.order_value / st.square_meters AS square_revenue
    FROM retails.stores st
    INNER JOIN total_revenue tr
        ON tr.store_key = st.store_key
    WHERE st.country != 'Online'
), quartile_rank AS (
    SELECT
        ra.store_key
        , ra.country
        , ra.square_revenue
        , NTILE(4) OVER( PARTITION BY ra.country ORDER BY ra.square_revenue DESC) AS quartile
    FROM revenue_acreage ra
)
SELECT
    qr.store_key
    , qr.country
    , qr.square_revenue
    , CASE qr.quartile
        WHEN 1 THEN 'Top 25%'
        WHEN 2 THEN 'Upper-middle'
        WHEN 3 THEN 'Lower-middle'
        ELSE 'Bottom 25%'
    END AS quartile_label
FROM quartile_rank qr
ORDER BY qr.store_key


Team marketing đang thiết kế recommendation engine, cần insight: những cặp sản phẩm nào hay được mua chung trong cùng 1 đơn hàng? Em lấy giúp top 20 cặp xuất hiện cùng nhau nhiều nhất, chỉ xét đơn có từ 2 sản phẩm khác nhau trở lên. Output: sản phẩm A, sản phẩm B, số lần xuất hiện cùng, tỷ lệ trên tổng số đơn.

In [14]:
WITH valid_orders AS ( 
    SELECT 
        DISTINCT s.order_number
        , COUNT( DISTINCT s.product_key) AS count_item
    FROM retails.sales s
    GROUP BY s.order_number
    HAVING COUNT( DISTINCT s.product_key) > 1
), product_pairs AS (
    SELECT 
        vo.order_number
        , s1.product_key AS product_1
        , s2.product_key AS product_2
    FROM retails.sales s1
    JOIN retails.sales s2
        ON s1.order_number = s2.order_number
    INNER JOIN valid_orders vo
        ON vo.order_number = s1.order_number
    WHERE s1.product_key < s2.product_key
), count_pair_product AS ( 
    SELECT
        pp.product_1
        , pp.product_2 
        , COUNT( DISTINCT pp.order_number) AS pair_count
        , ( 
            SELECT 
                COUNT( DISTINCT vo.order_number)
            FROM valid_orders vo
        ) AS count_all
    FROM product_pairs pp 
    GROUP BY pp.product_1, pp.product_2
)
SELECT 
    TOP 20
    cpp.product_1
    , cpp.product_2
    , cpp.pair_count
    , cpp.pair_count * 100.0 / cpp.count_all AS a
FROM count_pair_product cpp
ORDER BY pair_count DESC




(20 rows affected)

Total execution time: 00:00:00.471

product_1,product_2,pair_count,a
1648,1699,5,0.029203901641
101,417,4,0.023363121313
1630,2503,4,0.023363121313
421,434,4,0.023363121313
46,416,4,0.023363121313
430,435,4,0.023363121313
417,424,4,0.023363121313
432,440,4,0.023363121313
75,448,4,0.023363121313
97,444,4,0.023363121313
